<a href="https://colab.research.google.com/github/karenlaizsilva/visium-hd-spatial-transcriptomics/blob/main/notebooks/CRC/crc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Benchmarking de Algoritmos de Clustering em Transcriptômica Espacial: Visium HD em Câncer Colorretal e Espermatogênese

Karen Laiz da Silva

Trabalho de Conclusão de Curso apresentado para obtenção do título de especialista em Data Science & Analytics

Este notebook foi adaptado a partir do [guia oficial de análise da 10X Genomics](https://colab.research.google.com/github/10XGenomics/analysis_guides/blob/main/Visium_HD_multi_sample_comparison_python.ipynb), com customizações realizadas para os objetivos específicos deste TCC.

# **1. Configuração do ambiente e instalação de bibliotecas**

## **Python Libraries Used in this Analysis Guide**
This Analysis Guide will require the use of a few Python libraries. Some of the libraries are standard Python libraries like `gc` and `json`, but many of these libraries require installation.

* **[spatialdata](https://spatialdata.scverse.org/en/latest/index.html)** (as spd): A library for storing and manipulating multiomic spatial data. It can handle images, points, shapes (like polygons or circles), and tables (like `AnnData` objects). In the analysis, we import the `Identity` and `Scale` functions from `Spatialdata`'s `transformations` module.
* **[spatialdata_plot](https://github.com/scverse/spatialdata-plot)**: The plotting extension for `spatialdata`, enabling integrated visualization of images, shapes, and associated omics data.
* **[spatialdata_io](https://spatialdata.scverse.org/projects/io/en/latest/)** (as so): A sub-package of `spatialdata` designed for reading and writing various spatial omics file formats.
* **[geosketch](https://github.com/brianhie/geosketch)** (as sketch): A package used to downsample large datasets while preserving their underlying geometric structure and biological variability. This is a Python implementation of the algorithm described by *Hie et al. 2019*.
* **[numpy](https://numpy.org/)** (as np): This package provides support for large, multi-dimensional arrays and matrices, along with a collection of mathematical functions to operate on these arrays.
* **[pandas](https://pandas.pydata.org/)** (as pd): A library for data manipulation and analysis, providing data structures like DataFrames.
* **[scanpy](https://scanpy.readthedocs.io/en/stable/index.html)** (as sc): A toolkit for analyzing gene expression data installed with `Squidpy`, used for tasks such as preprocessing, dimensionality reduction (PCA, UMAP), clustering, and differential gene expression analysis.
* **scanpy.external** (as sce): A module within `Scanpy` that provides access to Harmonypy for batch correction. Harmonypy is a Python port of the R Harmony library used for dataset integration.
* **[json](https://docs.python.org/3/library/json.html)**: A standard Python library for importing and exporting JSON files. It is used in the creation of the `Spatialdata` object.
* **[gc](https://docs.python.org/3/library/gc.html)**: A standard Python library that provides an interface to the garbage collector that can help with memory management.
* **[geopandas](https://geopandas.org/en/stable/)** (as gpd): A library for working with geospatial vector data (like points, lines, and polygons).
* **spatialdata.models**: The functions imported from `spatialdata.models` library are used for defining and validating the structure and types of spatial data within the `Spatialdata` object.
* **[matplotlib.pyplot](https://matplotlib.org/stable/tutorials/pyplot.html)** (as plt): A Python package for data visualization.
* **[pydeseq2](https://pydeseq2.readthedocs.io/en/stable/)**: A Python implementation of the DESeq2 R library, used for differential gene expression analysis of count data. We import the `dds` and `ds` modules from the `pydeseq2` package.
* **[PIL](https://github.com/python-pillow/Pillow)**: A Python image processing package for opening, manipulating, and saving many images.
* **[shapely](https://pypi.org/project/shapely/)**: A Python package for manipulating and analyzing geometric objects. We import the `Polygon` module from the `shapely.geometry` module.

---

## **Installing Required Python Libraries to the Python Environment**
If you are not running the code using Google Colab, we recommend creating a fresh Python environment. In this guide, we use the Python package manager `uv` to install the required libraries.

To set up a local Python environment, follow these steps:

**Create a virtual environment (recommended):**

```python -m venv myenv```

**Activate the environment:**

```source myenv/bin/activate```


**Install required packages:**

```
(myenv) $ pip install --upgrade pip
(myenv) $ pip install  anndata==0.12.0 pydeseq2==0.5.2 squidpy==1.6.5 spatialdata==0.4.0 "spatialdata[extra]" geosketch==1.3 harmonypy==0.0.10 igraph==0.11.8
```

The Python script in this guide was written with Python v3.10 and the `spatialdata[extra]` package, which includes `spatialdata-io` v0.2.0 and `spatialdata-plot` v0.2.10.

If you are using Google Colab, execute each code cell to run the analysis.



#**1. Configuração do Ambiente e Instalação de Dependências**
Bibliotecas Utilizadas neste Estudo de Benchmarking
Para a análise dos dados de Câncer Colorretal e Espermatogênese via Visium HD, foram selecionadas bibliotecas de ponta no ecossistema de Ciência de Dados e Bioinformática. A escolha dessas ferramentas visa garantir a eficiência no processamento de grandes volumes de dados e a precisão na identificação de padrões espaciais.

Principais Bibliotecas e suas Aplicações:
SpatialData (spd): Framework central para manipulação de dados multiômicos espaciais. É responsável por integrar imagens de tecidos, coordenadas de pontos e tabelas de expressão gênica.

Scanpy (sc): O toolkit principal para análise de dados de células únicas. Neste projeto, é utilizado para o pré-processamento, redução de dimensionalidade (PCA, UMAP), clustering e análise de expressão diferencial.

GeoSketch (sketch): Essencial para este TCC, pois permite realizar o downsampling inteligente dos dados do Visium HD (que são massivos), preservando a variabilidade biológica e a estrutura geométrica dos tecidos sem sobrecarregar a memória RAM.

PyDESeq2: Utilizada para a análise estatística de expressão diferencial, permitindo identificar quais genes são estatisticamente significativos em cada cluster identificado.

GC: Biblioteca de Garbage Collector, utilizada para o gerenciamento proativo da memória RAM no ambiente do Google Colab, garantindo a estabilidade durante o processamento dos arquivos de alta resolução.

Bibliotecas de Suporte: Pandas e NumPy para manipulação de matrizes; Matplotlib e SpatialData-Plot para a geração de visualizações e gráficos espaciais.

Instalação do Ambiente de Execução
Para assegurar a reprodutibilidade deste estudo, as versões das bibliotecas foram fixadas. O processamento é realizado no ambiente de nuvem Google Colab, utilizando instâncias com aceleradores de hardware (GPU) e RAM configurada para alta demanda.

In [ ]:
%%capture

# Install required packages:
!uv pip install --upgrade pip
!uv pip install anndata==0.12.0 pydeseq2==0.5.2 squidpy==1.6.5 spatialdata[extra]==0.4.0 geosketch==1.3 harmonypy==0.0.10 igraph==0.11.8

In [ ]:
#instalação das bibliotecas

!pip install -q anndata==0.12.0
!pip install -q pydeseq2==0.5.2
!pip install -q squidpy==1.6.5
!pip install -q spatialdata==0.4.0
!pip install -q "spatialdata[extra]"
!pip install -q geosketch==1.3
!pip install -q harmonypy==0.0.10
!pip install -q igraph==0.11.8

print("instalação concluída")


In [ ]:
#Bibliotecas Base e Manipulação de Dados
import numpy as np
import pandas as pd
import json
import gc # Gerenciamento de memória RAM

#Framework de Transcriptômica Espacial
import spatialdata as spd
import spatialdata_plot as splt
import spatialdata_io as so
import scanpy as sc
import scanpy.external as sce

#Geometria e Processamento de Imagens
import geopandas as gpd
from spatialdata.models import Image2DModel, TableModel, ShapesModel
from shapely.geometry import Polygon
from PIL import Image

#Estatística e Amostragem
import geosketch as sketch
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

#Visualização
import matplotlib.pyplot as plt
from spatialdata.transformations import Identity, Scale

print("carregamento concluído")

#**2. ETL**


`create zarr`: função que automatiza a integração de diferentes fontes de dados provenientes do Visium HD em um só objeto para a análise espacial.

Entradas: Matriz de contagem (h5), imagem de alta resolução (png), fatores de escala (json) e segmentação de células (geojson).

Processamento:

1. Padroniza os sistemas de coordenadas;
2. Mapeia a identificação das células entre a matriz e o mapa geográfico (GeoJSON).
3. Cria objetos do tipo `SpatialData` para manipulação eficiente.

Saída: Um arquivo no formato .zarr, otimizado para leitura em nuvem e computação paralela.

-------------------
We use two custom helper function in this guide, which are defined in the next set of code cells.

The function `create_zarr` takes the raw output files from 10x Genomics' Visium HD processing and structures them into a single Zarr file, making the data ready for spatial analysis using libraries like `spatialdata`. It takes input paths, loads and prepares data, defines coordinate systems and transformations, processes the cell segmentation, integrates it with the `AnnData` object, creates `SpatialData` elements, and writes all of this to a Zarr file.

It takes five inputs:
* **`image_path`**: The path to the image file.
* **`count_matrix_path`**: The path to the cell segmentation filtered feature-barcode count matrix file.
* **`scale_factors_path`**: The path to the scale factors JSON file.
* **`geojson_path`**: The path to the cell segmentation GeoJSON file.
* **`sample_name`**: A name for the Zarr output file.

In this guide, we use the `hires_image.png` file to reduce the size of the Zarr file and the RAM requirement, which speeds up the code. The full-resolution microscope image can also be used, with adjustments to the code to properly scale the cell segmentation GeoJSON file. The images required in the final Zarr file will depend on the downstream analyses.



##Opção 1: Segmentação manual

Função desenvolvida manualmente para desenhar o formato exato de cada célula.

Vantagem: biologicamente mais preciso, pois respeita a morfologia do tecido.

Desvantagem: computacionalmente mais complexo e pesado, exige mais RAM.


In [ ]:
def create_zarr_manual_segmentation(count_matrix_path, image_path, scale_factors_path, geojson_path, sample_name):
    """
    ETL Manual: Ideal para máxima precisão biológica.
    Faz o mapeamento 1:1 entre GeoJSON e a matriz de contagem.
    """
    print(f"Iniciando processamento manual (segmentação): {sample_name}")

    # 1. Carregamento e Preparação
    adata = sc.read_10x_h5(count_matrix_path)
    adata.var_names_make_unique()
    adata.obs['sample'] = sample_name
    adata.obs.index = sample_name + "_" + adata.obs.index.astype(str)

    # 2. Processamento da Imagem
    image_data = np.array(Image.open(image_path))
    if image_data.ndim == 2:
        image_data = image_data[np.newaxis, :, :]
    elif image_data.ndim == 3:
        image_data = np.transpose(image_data, (2, 0, 1))

    # 3. Metadados e Geometrias
    with open(scale_factors_path, 'r') as f:
        scale_data = json.load(f)
    with open(geojson_path, 'r') as f:
        geojson_data = json.load(f)

    # 4. Sistemas de Coordenadas
    hires_scale = scale_data['tissue_hires_scalef']
    shapes_transformations = {"downscale_to_hires": Scale(np.array([hires_scale, hires_scale]), axes=("x", "y"))}
    image_transformations = {"downscale_to_hires": Identity()}

    # 5. Mapeamento GeoJSON -> AnnData
    geojson_features_map = {
        f"{sample_name}_cellid_{feature['properties']['cell_id']:09d}-1": feature
        for feature in geojson_data['features']
    }

    geometries = []
    cell_ids_ordered = []

    for obs_index_str in adata.obs.index:
        feature = geojson_features_map.get(obs_index_str)
        if feature:
            polygon_coords = np.array(feature['geometry']['coordinates'][0])
            geometries.append(Polygon(polygon_coords))
            cell_ids_ordered.append(obs_index_str)

    # 6. Estruturação Espacial
    shapes_gdf = gpd.GeoDataFrame({'cell_id': cell_ids_ordered, 'geometry': geometries}, index=cell_ids_ordered)
    adata.obs['cell_id'] = adata.obs.index
    adata.obs['region'] = sample_name + '_cell_boundaries'
    adata.obs['region'] = adata.obs['region'].astype('category')
    adata = adata[shapes_gdf.index].copy()

    # 7. Consolidação Zarr
    IMAGE_KEY = f"{sample_name}_hires_tissue_image"
    TABLE_KEY = 'segmentation_counts'
    SHAPES_KEY = f"{sample_name}_cell_boundaries"

    sdata = spd.SpatialData(
        images={IMAGE_KEY: Image2DModel.parse(image_data, transformations=image_transformations)},
        tables={TABLE_KEY: TableModel.parse(adata, region=SHAPES_KEY, region_key='region', instance_key='cell_id')},
        shapes={SHAPES_KEY: ShapesModel.parse(shapes_gdf, transformations=shapes_transformations)}
    )

    sdata.write(sample_name, overwrite=True)
    print(f"✅ Arquivo Zarr Manual para {sample_name} criado!")

    del sdata
    gc.collect()

##Opção 2: Square-Bins

Divide o tecido em bins. Ideal para análises exploratórioas rápidas em grandes datasets. Mais rápida e simplificada pois usa uma função pronta da biblioteca (´so.visium_hd´)



In [ ]:
def create_zarr_square_bins(path_to_outputs, zarr_name, bin_size):
    """
    ETL Simplificado: Alta performance.
    Ideal para datasets massivos de Câncer Colorretal.
    """
    print(f"Gerando Zarr para Square-bins: {zarr_name} (Bin: {bin_size})")

    sdata = so.visium_hd(
        path=path_to_outputs,
        load_all_images=True,
        bin_size=bin_size
    )

    sdata.write(zarr_name, overwrite=True)
    print(f"✅ Arquivo Zarr de Bins criado!")

    del sdata
    gc.collect()

##Opção 3: Segmentação Automática
Utiliza os recursos nativos e modernos da biblioteca para extrair automaticamente a segmentação de núcleos. Combina a precisão da segmentação com a facilidade de automação das ferramentas mais recentes.


In [ ]:
def create_zarr_auto_segmentation(path_to_outputs, zarr_name):
    """
    ETL Moderno: Utiliza a inteligência nativa do spatialdata-io
    para extrair formas nucleares sem código manual excessivo.
    """
    print(f"Processando segmentação automática: {zarr_name}")

    sdata = so.visium_hd(
        path=path_to_outputs,
        load_all_images=True,
        load_segmentations_only=True,
        load_nucleus_segmentations=True
    )

    sdata.write(zarr_name, overwrite=True)
    print(f"✅ Zarr de segmentação automática criado!")

    del sdata
    gc.collect()

Please note that using the `so.visium_hd` function will result in different names in the `SpatialData` object for the image, shapes, tables, and coordinate systems, requiring corresponding adjustments to the code.

The second helper function `crop0` ensures that the images generated from the analysis are cropped to the region of interest, aligning with the Visium HD Capture Area of each sample. It takes as input a `SpatialData` object (`x`), a target coordinate system (`crs`), and a bounding box dictionary (`bbox`). A bounding box dictionary is a way to represent a rectangular region in a 2D space using a Python dictionary. It typically contains the minimum and maximum coordinates for both the x and y axes that define the boundaries of the box. The function assumes that the `bbox` dictionary was created using `spd.get_extent` with the same coordinate system. Internally, the function calls `spd.bounding_box_query` and uses the minimum and maximum coordinates from the dictionary to subset the data from the `SpatialData` object that falls within this defined rectangle. This ensures that subsequent visualizations or analyses are focused only on the relevant part of the data. This is required because the microscope image is often larger than the Visium HD Gene Expression capture area.

In [ ]:
def crop0(x,crs,bbox):
    return spd.bounding_box_query(
        x,
        min_coordinate=[bbox['x'][0], bbox['y'][0]],
        max_coordinate=[bbox['x'][1], bbox['y'][1]],
        axes=("x", "y"),
        target_coordinate_system=crs,
    )

# **3. Download e Preparação dos Dados**

Os datasets utilizados neste estudo de benchmarking são disponibilizados publicamente pela 10x Genomics. Embora o pacote de dados contenha múltiplas amostras, este notebook foca especificamente na análise do Paciente 1 (CRC).

Dataset de Referência:
Visium HD, Sample P1 Colon Cancer (CRC): Tecido de câncer de cólon humano (Paciente 1).

Proveniência dos Dados:
Os dados foram originalmente processados utilizando o pipeline spaceranger count v3.0.0. No entanto, para gerar as saídas de segmentação celular utilizadas neste guia e no TCC, os datasets foram reprocessados com o spaceranger count v4.0.1.

O diretório de saída (outs) contém a matriz de contagem baseada em bins, a imagem de alta resolução e os arquivos de segmentação celular (cell_segmentation.geojson), essenciais para a nossa análise de alta resolução.

Nota Acadêmica: A utilização de dados públicos da 10x Genomics garante a reprodutibilidade dos resultados e permite o benchmarking contra pipelines padrão da indústria.

ATUALIZAR COM A MINHA DESCRIÇÃO DOS DADOS

### **Amostra analisada: Paciente 1 (CRC)**

In [ ]:
%%capture
!wget https://cf.10xgenomics.com/supp/spatial-exp/analysis-workshop/multisample_raw_data.tar.gz #download dos dados brutos
!tar xvzf multisample_raw_data.tar.gz #extração de arquivos
!rm multisample_raw_data.tar.gz #remoção do arquivo compactado para otimizar espaço
print("estraçao concluída]")

In [ ]:
from IPython.display import Image as ShowImage, display

caminho_imagem = 'data/Cancer_P1_tissue_hires_image.png'

print("Visualização da Amostra P1 - Câncer Colorretal:")
display(ShowImage(filename=caminho_imagem, width=600))

If you are working with your own data, for each dataset, the `outs` directory will contain the cell segmentation based binned output and spatial output data. For a more detailed description of Visium HD's outputs, see the documentation on our support [site](https://www.10xgenomics.com/support/software/space-ranger/latest/analysis/outputs/output-overview).

The data was originally processed using `spaceranger count` v3.0.0. However, to generate the Space Ranger cell segmentation outputs used in this guide, the public colon cancer and normal adjacent tissue datasets were reprocessed using `spaceranger count` v4.0.1.

The next code cell downloads and extracts the data required to make the spatial Zarr files and `SpatialData` objects.

#**4. Conversão para formato Zarr e criação do Objeto `SpatialData`**
Nesta etapa, transformamos as saídas padrão do Space Ranger em arquivos Zarr. O Zarr é o formato preferido da biblioteca spatialdata porque permite ler grandes volumes de dados de forma eficiente, carregando apenas o necessário para a memória RAM — algo crucial para o nosso benchmarking no Colab.

Abaixo, estruturamos um dicionário onde cada chave é o nome da amostra e o valor é uma lista contendo os caminhos para:

A matriz de contagem (.h5).

A imagem de alta resolução (.png).

O arquivo de fatores de escala (.json).

O arquivo de segmentação celular (.geojson).

O nome final do arquivo Zarr.

------
# **Section 3: Conversion of Space Ranger Output to Zarr Format and SpatialData Object Creation**

We begin this section by converting the standard Visium HD output into a Zarr file, as the `spatialdata` library expects data in this format. If you are running this code locally, this code needs to be run only once, as the datasets can be loaded directly from the saved Zarr files afterward.

In the following code snippet, a dictionary is created where each sample name serves as a unique key. The value associated with each key is a list containing:
* the path and filename of the filtered feature-cell matrix in `h5` format.
* the location and name of the image to be stored in the `SpatialData` object.
* the scale factors JSON file, so that the cell segmentation results can beoverlaid onto the tissue image.
* the cell segmentation GeoJSON file, so the cell segmentation results can be visualized on the tissue image.
* the desired name for the Zarr file.

Each key-value pair in this dictionary is then processed by the `create_zarr` helper function. For this specific example, we only use the `tissue_hires_image.png`. Other images, such as a high-resolution microscope image or CytAssist image, can be added to the `SpatialData` object.

##4.1. Processamento em Lote (Batch Processing)

REMOVER

O que isso muda no seu TCC?
Escalabilidade: Mesmo que seu foco seja o P1, mostrar que você processou o P2 (outro câncer) e os normais (P3 e P5) como controle dá muito mais robustez ao seu benchmarking.

Automação: No MBA, os avaliadores adoram ver que você sabe usar loops (for) e dicionários para automatizar tarefas repetitivas. Isso é puro "Analytics mindset".

Higiene de RAM: O comando gc.collect() no final é o seu seguro de vida. Ele avisa ao Colab: "Terminei essa parte pesada, pode limpar a memória para a próxima seção".

In [ ]:
# Criando o dicionário de amostras com os nomes REAIS dos arquivos da pasta /data
# Nota: Focaremos no Patient 1 (P1) e Patient 2 (P2) para o estudo de câncer.
samples = {
    "Colon_Cancer_P1": [
        "data/Cancer_P1_filtered_feature_cell_matrix.h5",
        "data/Cancer_P1_tissue_hires_image.png",
        "data/Cancer_P1_scalefactors_json.json",
        "data/Cancer_P1_cell_segmentations.geojson",
        "Colon_Cancer_P1"
    ],
    "Colon_Cancer_P2": [
        "data/Cancer_P2_filtered_feature_cell_matrix.h5",
        "data/Cancer_P2_tissue_hires_image.png",
        "data/Cancer_P2_scalefactors_json.json",
        "data/Cancer_P2_cell_segmentations.geojson",
        "Colon_Cancer_P2"
    ],
    "Colon_Normal_P3": [
        "data/Norm_P3_filtered_feature_cell_matrix.h5",
        "data/Norm_P3_tissue_hires_image.png",
        "data/Norm_P3_scalefactors_json.json",
        "data/Norm_P3_cell_segmentations.geojson",
        "Colon_Normal_P3"
    ],
    "Colon_Normal_P5": [
        "data/Norm_P5_filtered_feature_cell_matrix.h5",
        "data/Norm_P5_tissue_hires_image.png",
        "data/Norm_P5_scalefactors_json.json",
        "data/Norm_P5_cell_segmentations.geojson",
        "Colon_Normal_P5"
    ],
}

# Loop que percorre o dicionário e chama a função de ETL Manual (Opção 1)
for key, inputs in samples.items():
    create_zarr_manual_segmentation(
        count_matrix_path = inputs[0],
        image_path = inputs[1],
        scale_factors_path = inputs[2],
        geojson_path = inputs[3],
        sample_name = inputs[4]
    )

# Limpeza de memória para garantir estabilidade nas próximas etapas
del samples, inputs, key
gc.collect()

print("concluído")

##4.2. Arquitetura do Objeto SpatialData
Após o processamento de ETL, os dados são consolidados em um objeto do tipo SpatialData. Para o nosso benchmarking, é fundamental entender que este objeto funciona como um "contêiner" que sincroniza três camadas distintas:

Camada de Imagem: Armazena a histologia (H&E) em diferentes resoluções.

Camada de Formas (Shapes): Contém os polígonos resultantes da segmentação celular (GeoJSON).

Camada de Atributos (Table): Contém a matriz de expressão gênica e metadados das células (AnnData).

Insight de Engenharia: Esta estrutura permite que façamos consultas espaciais complexas (ex: "quais genes estão expressos apenas na borda do tumor?") de forma performática.

(SUBSTITUI IMAGEM GIGANTE DO TUTORIAL)Consolidação dos Datasets em um Objeto Único
Nesta etapa, carregamos os arquivos Zarr gerados individualmente e os consolidamos em um único objeto SpatialData. Isso facilita a análise comparativa entre as amostras de câncer e as amostras de controle (tecido normal).

Estrutura do Objeto SpatialData:
O SpatialData funciona como um "contêiner" inteligente que organiza:

.images: O contexto visual (Histologia H&E).

.shapes: A geometria das células (os polígonos que criamos no ETL).

.tables: A matriz AnnData com a expressão gênica. Nela temos:

.X: Matriz de contagem bruta.

.obs: Metadados das células (ex: a qual paciente a célula pertence).

.var: Metadados dos genes (nomes e IDs).

##**4.3. Consolidação dos Datasets em um Objeto Único**
Nesta etapa, carregamos os arquivos Zarr gerados individualmente e os consolidamos em um único objeto SpatialData. Isso facilita a análise comparativa entre as amostras de câncer e as amostras de controle (tecido normal).

Estrutura do Objeto SpatialData:
O SpatialData funciona como um "contêiner" inteligente que organiza:

.images: O contexto visual (Histologia H&E).

.shapes: A geometria das células (os polígonos que criamos no ETL).

.tables: A matriz AnnData com a expressão gênica. Nela temos:

.X: Matriz de contagem bruta.

.obs: Metadados das células (ex: a qual paciente a célula pertence).

.var: Metadados dos genes (nomes e IDs).

------------
Por que essa etapa é o "pulo do gato" do seu TCC?
Benchmarking Real: Ao concatenar os pacientes 1 e 2 (Câncer) com os 3 e 5 (Normal), você permite que seu algoritmo de clustering tente separar o que é tumor do que é tecido saudável de forma automática.

Eficiência de Memória: O uso do %%time no topo da célula é ótimo para o TCC. Você pode reportar no seu texto: "O processo de consolidação de X amostras levou Y minutos no ambiente Google Colab". Isso é um dado técnico valioso.

Segurança de Dados: Salvar o concatenated_sdata no disco é o seu "checkpoint". Se o Colab cair agora, você não precisa rodar o ETL de novo; basta ler esse arquivo Zarr final.
--------------


## **The SpatialData Object and Its Components**
The `spatialdata` library is built to manage and analyze multiomic spatial datasets. It brings together multiple data types into a single, unified `SpatialData` object. These objects act as on-disk containers that utilize the Zarr file format to store various **Elements** or data types.
A `SpatialData` object created from Visium HD data typically has the following Elements:
* **Images**: CytAssist and microscopy images (e.g., H&E, fluorescence) providing spatial context. These can be accessed via `sdata.images`.
* **Shapes**: Geometric annotations such as polygons or circles representing regions of interest, cells, or spots. In Visium HD, these often represent binned regions or cell segmentations. These are accessible via `sdata.shapes`.
* **Tables**: An `AnnData` object associated with the spatial elements, typically containing gene expression data, cellular metadata, and computational results (e.g., clusters, UMAP embeddings). This Element is used for downstream analyses and is accessed via `sdata.tables`. Each `AnnData` table within `sdata.tables` has:
  * `.X`: The primary data matrix (e.g., raw counts, normalized counts).
  * `.obs`: Observation metadata (e.g., sample ID, cluster assignments).
  * `.var`: Variable metadata (e.g., gene names).
  * `.obsm`: Multi-dimensional annotations (e.g., PCA, UMAP embeddings).
  * `.layers`: Alternative representations of `.X`.

In the next code block, the `SpatialData` object is created from the Zarr files. Each Zarr file is read into a list of `SpatialData` objects using the `read_zarr` function before the objects are concatenated. In addition, a `sample` column is added to each `AnnData` table within the `SpatialData` object, and the `var_names_make_unique` function is used to ensure that gene names are unique.


In [ ]:
%%time

# 1. Mapeamento dos caminhos dos arquivos Zarr que criamos na Seção 4
# Certifique-se de que os nomes batem com o que foi salvo no disco
visium_hd_zarr_paths = {
    "Cancer_P1": "./Colon_Cancer_P1",
    "Cancer_P2": "./Colon_Cancer_P2",
    "Normal_P3": "./Colon_Normal_P3",
    "Normal_P5": "./Colon_Normal_P5"
}

sdatas = []

print("📂 Carregando amostras para a memória...")

for key, path in visium_hd_zarr_paths.items():
    # Lendo o arquivo Zarr do disco
    sdata = spd.read_zarr(path)

    # Housekeeping: Garantindo nomes de genes únicos e identificando a amostra
    for table in sdata.tables.values():
        table.var_names_make_unique()
        table.obs["sample"] = key # Isso permite filtrar por paciente depois

    sdatas.append(sdata)

    # Limpeza rápida de memória por iteração
    del sdata
    gc.collect()

# 2. Concatenação: Unindo todas as amostras em um único objeto global
print("🔗 Concatenando amostras... (Isso pode levar um tempo devido ao volume de dados)")
concatenated_sdata = spd.concatenate(sdatas, concatenate_tables=True)

# 3. Salvando o objeto consolidado para não precisar rodar tudo de novo depois
concatenated_sdata.write("concatenated_sdata", overwrite=True)

# Limpeza final para liberar RAM
del sdatas, visium_hd_zarr_paths
gc.collect()

# 4. Recarregando o objeto consolidado (Boa prática para garantir integridade)
concatenated_sdata = spd.read_zarr("concatenated_sdata")

print("-" * 33)
print("✅ Objeto consolidado criado com sucesso!")
print(concatenated_sdata)

Sucesso total, Karen! Esse "log" que você postou é o troféu da sua engenharia de dados. Você acaba de criar uma base de dados com 652.256 células e 18.085 genes. Para uma Analista de CRM, pense nisso como uma base de 650 mil clientes, cada um com 18 mil atributos. É um "Big Data" de respeito para o seu TCC!

**Descrição do Objeto Consolidado**

O objeto concatenated_sdata agora atua como nossa base de dados centralizada para o benchmarking. Ele é composto por:

Imagens (4 unidades): As fotos de histologia (hires_tissue_image.png) de cada amostra (P1, P2, P3 e P5), devidamente alinhadas em seus respectivos sistemas de coordenadas.

Formas / Shapes (4 unidades): As representações geométricas (polígonos) das células segmentadas, essenciais para a visualização espacial dos resultados de clustering.

Tabela (1 unidade): Uma tabela unificada (segmentation_counts) que armazena a expressão gênica de todas as 652.256 células processadas, permitindo análises estatísticas em lote.

Nota sobre Coordenadas: Estamos utilizando o sistema downscale_to_hires. Isso garante que, quando filtrarmos um gene na tabela, ele "caia" exatamente no lugar certo em cima da imagem da célula na histologia.


-----------------------



The concatenated `SpatialData` object (`concatenated_sdata`) contains:
* four images (the `hires_tissue_image.png` for each sample), each with its corresponding coordinate system.
* four shapes, which are `GeoDataFrame` representations of the cell segmentation bins.
* one table that holds the cell segmentation bin gene expression data along with its metadata.

Using Sample P1 CRC as an example, the `SpatialData` object contains the following Elements:
* **Images**:
  * `Colon_Cancer_P1_hires_tissue_image`: This is the high-resolution version of the downsampled PNG image, associated with the `downscale_to_hires` coordinate system.
* **Shapes**:
  * `Colon_Cancer_P1_cell_boundaries`: This represents the spatial information (shape and location) for the cell segmentation bins.
* **Tables**:
  * `segmentation_counts`: This table contains the gene expression data and associated metadata for the cell segmentation bins.

In this guide, the hires PNG images from the Visium HD output folder are used with the `downscale_to_hires` coordinate system. For more information on `SpatialData` objects, see `scverse`'s introductory [tutorials](https://spatialdata.scverse.org/en/stable/tutorials/notebooks/notebooks/examples/intro.html).


# **Section 4: Quality Control and Filtering**

The analysis begins by removing low-quality cell segmentation bins from the datasets. In this example, this is done by visualizing the total UMI distribution to estimate a suitable cutoff for empty or sparsely populated bins. Even though mitochondrial content is calculated, it is not used as a filtering criterion, as high levels of mitochondrial genes can be biologically relevant in spatial data.



------


Seção 6: Controle de Qualidade (QC) e Filtragem
Nesta etapa, aplicamos filtros rigorosos para garantir que a análise subsequente seja baseada em dados biológicos reais e não em ruído técnico. O controle de qualidade (QC) é essencial para o benchmarking, pois algoritmos de clustering são sensíveis a outliers e dados de baixa qualidade.

Métricas de Filtreagem:
Total Counts: Número total de moléculas de RNA detectadas por célula.

Number of Genes: Quantidade de genes únicos expressos.

Mitochondrial Percentage: Proporção de genes mitocondriais (indicador de estresse ou morte celular).

In [ ]:
# Acessando a tabela de contagens (AnnData) dentro do objeto consolidado
adata = concatenated_sdata.tables['segmentation_counts']

# Identificando genes mitocondriais (geralmente começam com 'MT-')
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Calculando métricas de QC usando a função do Scanpy
print("📊 Calculando métricas de QC...")
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    inplace=True,
    percent_top=[50, 100, 200, 500],
    log1p=False
)

print(f"✅ Métricas calculadas para {adata.n_obs} células.")

# Visualizando as primeiras linhas das métricas calculadas
adata.obs.head()

In [ ]:
# 1. Recalculando métricas com log1p para as visualizações
# O log1p ajuda a visualizar dados com alta dispersão (comum em transcriptômica)
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True, log1p=True)

# 2. Visualização: Total de UMIs (Moléculas de RNA) por Amostra
# As linhas vermelhas em y=4 e y=8 representam os limites sugeridos de corte
plt.figure(figsize=(10, 5))
sc.pl.violin(adata, keys=["log1p_total_counts"], groupby="sample", stripplot=False, inner="box", show=False)
plt.axhline(y=4, color='r', linestyle='--', label="Limite Inferior")
plt.axhline(y=8, color='r', linestyle='--', label="Limite Superior")
plt.title("Distribuição de Total Counts (Log1p) por Amostra")
plt.show()

# 3. Visualização: Quantidade de Genes por Amostra
plt.figure(figsize=(10, 5))
sc.pl.violin(adata, keys=["log1p_n_genes_by_counts"], groupby="sample", stripplot=False, inner="box", show=False)
plt.title("Distribuição de Genes Únicos (Log1p) por Amostra")
plt.show()

# 4. Visualização: Genes Mitocondriais (Indicador de Morte Celular)
plt.figure(figsize=(10, 5))
sc.pl.violin(adata, keys=["pct_counts_mt"], groupby="sample", stripplot=False, inner="box", show=False)
plt.title("Percentual de Genes Mitocondriais por Amostra")
plt.show()

plt.close('all')

Based on the log1p total UMI violin plot, cell segmentation bins with fewer than 53 counts (corresponding to a log1p value of 4) and more than 2,979 counts (corresponding to a log1p value of 8) are removed from the datasets using `Scanpy’`s `filter_cells` function. Additionally, genes present in fewer than 50 cell segmentation bins are excluded from the analysis using `Scanpy’`s `filter_genes` function. It is important to note that both the count cutoffs and the number of bins a gene must be present in will vary between experiments and should be determined empirically.

In this example, we filtered all the datasets using the same criteria. However, with a different set of datasets, the samples may need to be filtered using sample-specific criteria.



In [ ]:

# Estimating the cut off
min_counts = np.expm1(4).astype("int")
max_counts = np.expm1(8).astype("int")

# Filtering genes and cells
sc.pp.filter_genes(adata, min_cells=50)
sc.pp.filter_cells(adata, min_counts=min_counts)
sc.pp.filter_cells(adata, max_counts=max_counts)

# Visualization for QC
sc.pl.violin(adata=adata, keys=["log1p_total_counts"], stripplot=False, inner="box",show=False, groupby="sample",)
plt.title("Total UMI by Sample")
plt.axhline(y=4, color='r', linestyle='-')
plt.axhline(y=8, color='r', linestyle='-')
plt.title("Total UMI by Sample")
plt.show()

sc.pl.violin(adata=adata, keys=["log1p_n_genes_by_counts"], groupby="sample", stripplot=False, inner="box",show=False)
plt.title("Total Genes by Sample")
plt.show()

sc.pl.violin(adata=adata, keys=["log1p_total_counts_mt"], groupby="sample", stripplot=False, inner="box",show=False)
plt.title("Mitochondrial Genes by Sample")
plt.show()

plt.close('all')

# storing filtered counts
adata.layers["filtered_counts"] = adata.X.copy()


del max_counts, min_counts
gc.collect()


*Note: A copy of the raw count data is saved in the filtered_counts layer for later use. It's crucial to use the `.copy()` method to ensure the raw count values are preserved.*


# **Section 5: Data Normalization and Dimensionality Reduction**
Now that the `SpatialData` object is filtered, we normalize the filtered counts and perform Principal Component Analysis (`PCA`) for dimensionality reduction using `Scanpy`'s `normalize_total`, `log1p`, and `pca` preprocessing functions. Before running PCA, there is an option to identify highly variable genes using `Scanpy`'s `highly_variable_genes` function, so that only these genes are used in the PCA. However, in practice, we prefer to use the entire gene set. You may wish to find highly variable genes and use them in your own analysis. In this analysis, we set the `target_sum` to `None`, so that after normalization, each cell segmentation bin has a total UMI count equal to the median of the total UMI count for all cell segmentation bins before normalization. By default, `Scanpy’`s `pca` function generates the first 50 principal components (PCs). The normalization and dimensionality reduction steps are performed on the `SpatialData` object without copying the `AnnData` object. The results are stored directly within the `sdata_concatenate` object as the `adata` variable is linked to it.

We also generate an Elbow plot of the data to ensure that sufficient variation is captured in downstream analyses. `Scanpy`'s functions, by default, will use all generated PCs unless specified otherwise. However, you may choose to restrict the number of PCs to prevent overfitting the data.


In [ ]:
%%time

sc.pp.normalize_total(adata, target_sum = None)
sc.pp.log1p(adata)
sc.tl.pca(adata)

adata.write("preprocessed_adata.h5ad")

# Elbow plot
sc.pl.pca_variance_ratio(adata, log=True,n_pcs=50)

# **Section 6: Clustering and UMAP Visualization**
Now that we have standardized the data and performed a PCA, we will cluster and visualize the results. `Scanpy`'s `neighbors` function generates a neighbor distance matrix and a neighborhood graph, which is used by `Scanpy`'s `leiden` function to cluster the data. Finally, `Scanpy`’s `umap` function is used to visualize the results.

## Extra Options
When running `Scanpy`'s `neighbors` function, the distance metric selected will impact the clustering results. Therefore, you may need to explore different metrics depending on your datasets. Common distance metrics available in `Scanpy` include:
* **Euclidean distance**: The straight-line distance between two points in multi-dimensional space. It tends to group cells with similar overall expression magnitudes. If some genes have very high expression, they can dominate the distance calculation.
* **Manhattan distance (L1 distance, City Block distance)**: The sum of the absolute differences of their coordinates. It is less sensitive to outliers than Euclidean distance. It is useful when differences in individual features are more important than overall magnitude.
* **Cosine distance/similarity**: Measures the angle between two vectors. A smaller angle (closer to 0) indicates higher similarity. It focuses on the orientation of the expression profiles, rather than their magnitude. This is particularly useful when the relative proportions of gene expression might be more informative than the absolute counts. Cell segmentation bins expressing the same genes in similar proportions will be considered close, even if one has a higher total count.
* **Correlation-based distances (e.g., Pearson, Spearman)**: These typically define distance as `1 - correlation_coefficient`. Cell segmentation bins are considered similar if their gene expression profiles are highly correlated, regardless of absolute expression values. This method identifies cell segmentation bins with similar patterns of gene activity.

A metric that emphasizes magnitude (like Euclidean) might connect cells based on overall transcriptional activity, while a metric emphasizing shape (like Cosine) might connect cells with similar gene expression patterns, even if their total RNA content differs.

For this analysis, we used `Scanpy`'s correlation distance metric, the clustering resolution (RES) is set to 0.8, and the number of neighbors is set to 15. These parameters will likely need to be fine-tuned for new analyses: a smaller resolution generally leads to fewer clusters, while increasing the number of neighbors will have a similar effect.

In addition, for optimal visualization, you may need to adjust the `min_dist` and `spread` parameters used in `Scanpy`'s umap function for Visium HD Gene Expression data. `min_dist` controls how tightly packed the points are in the final embedding, while `spread` determines the overall scale and the separation between clusters. These values will need to be empirically determined.

_Note: This is a time-consuming (20-30 min) process._

In [ ]:
%%time

# neighborhood and clustering resolution
RES = 0.5 # clustering resolution
NEIGHBORS = 30  # number of neighbors

MIN_DIST=0.5 #default 0.5
SPREAD=2 #default 1

sc.pp.neighbors(adata, n_neighbors=NEIGHBORS, use_rep="X_pca",metric="correlation")
sc.tl.leiden(adata, flavor="igraph", key_added="clusters", resolution=RES,random_state=0)

# To ensure that the results are reproducible we are going to reorder the clusters by size.
adata.obs['orig_clusters'] = adata.obs['clusters']

clusters = adata.obs['clusters'].astype(int)

# Count cells per cluster
cluster_sizes = clusters.value_counts().sort_values(ascending=False)

# Create mapping: old cluster ID → new ordered ID
cluster_order = {old: new for new, old in enumerate(cluster_sizes.index)}

# Relabel clusters in adata
adata.obs['clusters'] = clusters.map(cluster_order).astype(str)

# Set random_state for reproducible UMAP
sc.tl.umap(adata,min_dist=MIN_DIST, spread=SPREAD, random_state=0)

# Plot UMAP
sc.pl.umap(adata, color=["clusters"], title="UMAP by Clusters")
sc.pl.umap(adata, color=["sample"], title="UMAP by Sample")

# Cell distribution across clusters
sample_names = adata.obs["sample"].unique()
plt.imshow(pd.crosstab(adata.obs["sample"], adata.obs["clusters"]), cmap='hot', interpolation='nearest')
plt.title("Cell Distribution Across Clusters")
plt.xlabel("Cluster")
plt.yticks(range(len(sample_names)), sample_names)
plt.show()
plt.close('all')

del RES, NEIGHBORS, MIN_DIST, SPREAD
gc.collect()


We color the UMAP plot by sample identity to quickly check for any batch effects. There is not a clear separation in the sample clusters, as seen in the Analysis Guide, [Correcting Batch Effects in Visium Data](https://www.10xgenomics.com/analysis-guides/correcting-batch-effects-in-visium-data). In the above code block, we also generate a heatmap of the cross-tabulation between sample and cluster.

`pd.crosstab` creates a cross-tabulation (a frequency or contingency table) that counts the number of occurrences for each combination of categories—in this case, the samples and the clusters. The resulting table has samples as rows and clusters as columns, with each cell containing the count of cells that belong to a specific sample-cluster pair.

The heatmap generated from the cross-tabulation shows the frequency of cells in each cluster for each sample.

- **Rows:** Represent the different samples in your dataset.
- **Columns:** Represent the different clusters identified.
- **Color Intensity:** The color of each square indicates the number of barcodes. Darker colors (e.g., black) mean a low number of cells for that sample-cluster combination. Brighter colors (e.g., white or bright yellow) mean a high number of cells.

If the colors are relatively uniform across the rows, it suggests that each cluster is well-represented in all samples. This indicates that the clusters are not simply artifacts of a specific sample.

A bright spot in one row (sample) but not in others indicates that a particular cluster is highly enriched or unique to that specific sample. This could be due to a biological difference between samples or a batch effect. We know from the differential gene expression analysis that the bright spots in these datasets are tumor.

If a column is uniformly dark, it means that cluster is not present in any of the samples. If only one or two rows are dark, it suggests that cluster is absent in those specific samples.

In this particular dataset, a clear batch effect is not observed. However, in the next section, we demonstrate how `Harmony` can correct for batch effects if they were present. We also demonstrate how applying batch correction when a batch effect is not present can over-correct the data.

# **Section 7: Batch Correction (Optional)**
Sometimes a batch effect is present in the data. If one is observed, then `Harmony` can be used to correct the PCA embeddings to remove it. In the code, `Scanpy`'s preprocessing `harmony_integrate` function calls a Python implementation of the Harmony algorithm to perform this correction. For details on the algorithm, see the documentation for the R library implementation of [Harmony](https://cran.r-project.org/web/packages/harmony/vignettes/quickstart.html).

The sample column in the `AnnData` object is used as a key for the `harmony_integrate` function to differentiate between the various samples. Alternatively, if samples were processed in different batches, a dedicated batch column in the `AnnData` object could be used to group them by their processing batch. After the Harmony algorithm runs, the new PCA embeddings are copied over to the `X_pca` embeddings, and the original clustering results are saved. Subsequently, `Scanpy`'s `neighbors`, `leiden`, and `umap` functions are called to process the batch-corrected data.


In [ ]:
%%time

# neighborhood and clustering resolution
RES = 0.5 # clustering resolution
NEIGHBORS = 30  # number of neighbors

MIN_DIST=0.5 #default 0.5
SPREAD=2 #default 1

# Performing batch correction
adata_harmony = concatenated_sdata["segmentation_counts"].copy()
sce.pp.harmony_integrate(adata_harmony, key="sample", basis="X_pca",max_iter_harmony=20)

# Copying the harmony PCA embedding results
adata_harmony.obsm["X_pca_orig"] = adata_harmony.obsm["X_pca"]
adata_harmony.obsm["X_pca"] = adata_harmony.obsm["X_pca_harmony"]
adata_harmony.obs["cluster_orig"] =  adata_harmony.obs["clusters"]

sc.pp.neighbors(adata_harmony, n_neighbors=NEIGHBORS, use_rep="X_pca",metric="correlation")
sc.tl.leiden(adata_harmony, flavor="igraph",key_added="harmony_clusters", resolution=RES,random_state=0)
adata_harmony.obs["clusters"] =  adata_harmony.obs["harmony_clusters"]

# Set random_state for reproducible UMAP
sc.tl.umap(adata_harmony,min_dist=MIN_DIST, spread=SPREAD, random_state=0)

# Plot UMAP
sc.pl.umap(adata_harmony, color=["clusters"], title="Harmony Corrected UMAP by Clusters")
sc.pl.umap(adata_harmony, color=["sample"], title="Harmony Corrected UMAP by Sample")

# Cell distribution across clusters
sample_names = adata_harmony.obs["sample"].unique()
plt.imshow(pd.crosstab(adata_harmony.obs["sample"], adata_harmony.obs["clusters"]), cmap='hot', interpolation='nearest')
plt.title("Cell Distribution Across Clusters")
plt.xlabel("Cluster")
plt.yticks(range(len(sample_names)), sample_names)
plt.show()
plt.close('all')


# If you want to use the harmony results in the analysis then overwrite the AnnData table in the SpatialData object by removing the comment below.
# concatenated_sdata["segmentation_counts"] = adata_harmony

del adata_harmony, RES, NEIGHBORS, MIN_DIST, SPREAD
gc.collect()

`Harmony` does improve the alignment of the sample clustering and the cell distribution across the clusters, as seen in the uniformity of the heatmap color. In fact, the tumor clusters are collapsed into one cluster. However, when the `Harmony-corrected` `AnnData` object is used in downstream analyses, cell segmentation bin annotation is less accurate. The data is over-corrected, which can occur when samples that do not have a true batch effect are corrected, as the biology is removed from the analysis. Therefore, for this case, we used the uncorrected embeddings for the remaining analyses.


# **Section 8: Spatial Visualization of Clusters**

Now that we have clustered the data, the next step is to annotate the clusters. We begin by visualizing the clustering results directly on the microscope image.

To ensure only the relevant portion of the image (the area of the bins) is plotted, we use the `crop0` helper function. The bounding box (`bbox`) for the bin-associated area is created using `Spatialdata`'s `get_extent` function, which provides a dictionary containing the minimum and maximum `x` and `y` coordinates. This cropping step is important because the microscope image often captures a larger tissue area than the CytAssist's Capture Area.

Finally, the cluster results are plotted using the `SpatialData` object's `render_image` and `render_shape` methods.


In [ ]:
image_elements = list(concatenated_sdata.images.keys())
shape_elements = list(concatenated_sdata.shapes.keys())

# We are going to create a bounding box to crop the data to the capture area.
extents = []

for i in range(len(image_elements)):
    extent =  spd.get_extent(concatenated_sdata,elements=[shape_elements[i]],coordinate_system='downscale_to_hires')
    extents.append(extent)

# Plotting
if len(image_elements) != len(shape_elements):
    print("Check the spatial data to make sure that for every image there is a shape")
else:
    for i in range(len(image_elements)):
        print("Plotting: "+ image_elements[i])
        title=image_elements[i].replace("_hires_tissue_image","")
        crop0(concatenated_sdata,crs="downscale_to_hires",bbox=extents[i]).pl.render_images(image_elements[i]).pl.render_shapes(shape_elements[i],color="clusters").pl.show(coordinate_systems="downscale_to_hires", title=title)


For additional resources on plotting `SpatialData` objects, see `Spatialdata`'s Visium HD technology-focused [tutorial](https://spatialdata.scverse.org/en/latest/tutorials/notebooks/notebooks/examples/technology_visium_hd.html).

# **Section 9: Marker Gene Identification and Cluster Annotation**

To annotate the clusters, we can utilize canonical marker genes, or determine cluster-specific marker genes and then analyze the upregulated genes. We begin by using `Scanpy`'s `dotplot` function to visualize cell type markers in the samples, grouped by their respective clusters within the `AnnData` object. A third option is to integrate an annotated single-cell RNA-seq dataset to label the clusters, which is beyond the scope of this guide. In this analysis, we are considering only a subset of cells that can be present. To identify other cell types such as T cells, we will likely need a finer clustering resolution.

In [ ]:
# cannonical markers for annotation
marker_genes = {
    "Fibroblasts": ["COL1A1", "MMP2", 'VIM'],
    "Goblet cells": ["FCGBP", "MUC2", "CLCA1"],
    "Enterocyte":["EPCAM","KRT8","KRT20","FABP2"],
    "Plasma B cells":['JCHAIN','IGKC','IGHG1'],
    "B cell":['MS4A1','CD74','CD19','CD22'],
    "Smooth muscle":['TAGLN','DES','MYH11'],
    "Tumor":["CEACAM6",'REG1B',"REG1A"]
}

# Plot dotplot for initial cluster assessment

sc.pl.dotplot(adata = concatenated_sdata["segmentation_counts"], var_names = marker_genes, groupby="clusters", standard_scale="var")


For example, using the canonical markers, we observe that Cluster 3 expresses smooth muscle markers.

Cluster annotation can often be challenging when only canonical markers are used. To assist in this process, we can use `Scanpy`’s `rank_genes_groups` function to identify marker genes for each cluster. The results can be ranked by marker score or by the log fold-change. The top-ranked genes within each cluster can then be further analyzed using tools like [Enrichr](https://maayanlab.cloud/Enrichr/) to infer the cluster's potential cell type.

_This step can take 10-20 mins to run._


In [ ]:
%%time

# Obtain cluster-specific marker genes
sc.tl.rank_genes_groups(adata = concatenated_sdata["segmentation_counts"], groupby="clusters", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata = concatenated_sdata["segmentation_counts"], groupby="clusters", standard_scale="var", n_genes=5)
df_marker_genes = sc.get.rank_genes_groups_df(adata = concatenated_sdata["segmentation_counts"],group = None,pval_cutoff=0.05)
df_marker_genes.to_csv("marker_genes_pval.csv")

Once the clusters have been annotated, we can map these annotations to the clusters. For subsequent analyses in this guide, we will start by merging some of these clusters. We do this for illustrative purposes and to simplify the initial analyses. These merged clusters can always be further subdivided later to gain deeper insight into the biology of individual clusters. In the following code, we map the cell type annotations to the clusters, and then overlay the results onto the microscope image to visually confirm that the cluster annotations match the tissue morphology.


In [ ]:
# Cluster annotation mapping
original_clusters = concatenated_sdata["segmentation_counts"].obs['clusters']

cell_annotation = {
    '0': 'Fibroblast',
    '1': 'Tumor',
    '2': 'Plasma cell',
    '3': 'Smooth muscle',
    '4': 'Tumor',
    '5': 'B cell and T cell',
    '6': 'Enterocyte',
    '7': 'Plasma cell',
    '8': 'Goblet',
    '9': 'Enterocyte'
}

# Apply the mapping. This new_categories Series should have the same index as original_clusters.
new_categories = original_clusters.astype('string').map(cell_annotation)

# Assign to sdata_concatenate.
concatenated_sdata["segmentation_counts"].obs["grouped_clusters"] = new_categories.astype('category')

# Plotting with new grouped clusters
for i in range(len(image_elements)):
  print("Plotting: "+ image_elements[i])
  title=image_elements[i].replace("_hires_tissue_image","")
  crop0(concatenated_sdata,crs="downscale_to_hires",bbox=extents[i]).pl.render_images(image_elements[i]).pl.render_shapes(shape_elements[i],color="grouped_clusters").pl.show(coordinate_systems="downscale_to_hires", title=title)

We can confirm the identity of the clusters by examining the morphology of the microscope image. For example, the smooth muscle clusters overlay the smooth muscle visible in the microscope image.


In [ ]:
spd.bounding_box_query(
        concatenated_sdata,
        min_coordinate=[3235, 1500],
        max_coordinate=[4000, 1759],
        axes=("x", "y"),
        target_coordinate_system='downscale_to_hires').pl.render_images("Colon_Cancer_P2_hires_tissue_image").pl.show(coordinate_systems='downscale_to_hires', title="Colon_Cancer_P2")

spd.bounding_box_query(
        concatenated_sdata,
        min_coordinate=[3235, 1500],
        max_coordinate=[4000, 1759],
        axes=("x", "y"),
        target_coordinate_system='downscale_to_hires').pl.render_images("Colon_Cancer_P2_hires_tissue_image").pl.render_shapes(shape_elements[1],color="grouped_clusters").pl.show(coordinate_systems='downscale_to_hires',title="Colon_Cancer_P2")

The clustering results can also be written to a `CSV` file for import into Loupe Browser (v9.0.0 or later).

In [ ]:

for sample_name in concatenated_sdata["segmentation_counts"].obs['sample'].unique():
    # Filter the table for the current sample
    adata_sample = concatenated_sdata["segmentation_counts"][concatenated_sdata["segmentation_counts"].obs['sample'] == sample_name].copy()

    # Create a DataFrame
    df_output = pd.DataFrame({
        'Barcode': 'cellid_' + adata_sample.obs.index.str.split('cellid_').str[1],
        'Grouped_Annotation': adata_sample.obs['grouped_clusters']
    })

    # Save the results
    output_filename = f"{sample_name}_cell_clusters.csv"
    df_output.to_csv(output_filename, index=False)

    print(f"Saved {output_filename}")

del adata_sample, df_output, sample_name, output_filename
gc.collect()

# **Section 10: Differential Gene Expression Analysis**

Now that the clusters are annotated, we will pseudobulk the data to perform differential gene expression analysis using `DESeq2` on these aggregated counts. We will focus on the fibroblast cluster, aiming to identify genes that are differentially expressed between the `Cancer` and `Normal Adjacent` samples.

To achieve this, we use `Scanpy`’s `aggregate` function to group the `AnnData` object table by the `grouped_clusters` and `sample` in the `AnnData` object metadata. We use the `filtered_counts` layer for gene expression and sum the counts, as `DESeq2` requires raw count data as its input.

_Note: There are only n=2 biological replicates per condition. For greater statistical power, there should be more replicates. The minimum is n=3 and some studies recommend n=5 or more._


In [ ]:
# aggregate by the cluster and the sample id.
aggregated = sc.get.aggregate(concatenated_sdata["segmentation_counts"], by=["grouped_clusters","sample"], func=["sum"],layer="filtered_counts")

#  We will focus on one cluster of interest
aggregated_cluster_of_interest = aggregated[aggregated.obs["grouped_clusters"]== 'Fibroblast']


# Creating metadata table
metadata_df = aggregated_cluster_of_interest.obs["sample"].str.split("_", expand=True)
metadata_df.columns = ["tissue", "patient"]
metadata_df

#Creating counts table
counts_df = pd.DataFrame(aggregated_cluster_of_interest.layers["sum"],index=metadata_df.index,columns=aggregated_cluster_of_interest.var_names)
counts_df = counts_df.astype(int)

`DESeq2` requires three primary inputs: a metadata DataFrame and a counts DataFrame, along with a design formula.

* The metadata DataFrame maps each sample to its experimental conditions or covariates. In the metadata DataFrame, each row is a sample and each column is a condition or covariate.
* The counts DataFrame contains raw gene counts, with each row representing a sample and each column representing a gene.
* The design formula is the model `DESeq2` uses to estimate gene expression changes (log2 fold changes) and assess their statistical significance.

The design formula specifies which factors or covariates from the metadata DataFrame influence gene expression and how they relate to the observed counts. Written in R's formula syntax, it typically begins with a tilde `~` followed by the variables from the metadata table that you intend to include in the model. For this analysis, we use the formula `~tissue`, which will compare the fibroblasts in the `Cancer` and `Normal Adjacent` samples.

For more detailed information on `DESeq2` and setting up designs for various studies, see the official `DESeq2` R [documentation](https://bioconductor.org/packages/devel/bioc/vignettes/DESeq2/inst/doc/DESeq2.html#quick-start) and Python [documentation](https://pydeseq2.readthedocs.io/en/stable/).

In this example, the metadata table has two columns, one for the tissue type and the other for the patient identifier. Each row corresponds to a sample.


In [ ]:
print(metadata_df)

Similarly, we see in the counts table that each row is a sample and the columns are now genes.

In [ ]:
print(counts_df)

In the next section of the code, `DESeq2`is run on the samples. The code then generates a summary table of the results, specifically comparing fibroblast expression in `Cancer` samples to `Normal Adjacent` tissue samples.


In [ ]:
# Determining differentially expressed genes from the aggregated data using DeSeq2
dds = DeseqDataSet(
    counts = counts_df,
    metadata = metadata_df,
    design = "~tissue",
    refit_cooks=True
)

dds.deseq2()
ds = DeseqStats(dds, contrast=["tissue", "Cancer", "Normal"])
ds.summary()

print(ds.results_df.sort_values(by='log2FoldChange', ascending=False).head(10))
ds.results_df.to_csv('aggregated_diffexp.csv')

# Discussion
We saved the differential gene expression table as `aggregated_diffexp.csv`. These results can be ranked by metrics such as log2 Fold Change (`log2FoldChange`) or the adjusted p-value (`padj`). When ranking by adjusted p-value, the top ten upregulated genes are: SFRP4, INHBA, ADAM12, COMP, COL1A1, COL8A1, COL12A1, ITGA11, ITGBL1, and CTHRC1.

In [ ]:
# --- Filtering and printing the table ---
positive_fold_change_genes = ds.results_df[ds.results_df['log2FoldChange'] > 0]

top_10_positive_genes_table = positive_fold_change_genes.sort_values(by='padj', ascending=True).head(10)

print("\n--- Top 10 Positively Differentially Expressed Genes (Cancer vs. Normal) ---")
print(top_10_positive_genes_table)


Upregulation of these genes in fibroblasts contributes to colon cancer progression by remodeling the extracellular matrix (ECM), secreting pro-tumorigenic factors, and fostering an immunosuppressive and pro-invasive tumor microenvironment.

### Extracellular Matrix (ECM) Remodeling and Adhesion

These genes are directly involved in building, modifying, and interacting with the ECM.

* **COL1A1, COL8A1, COL12A1:** These genes encode different types of collagen, a major structural protein of the ECM. **Cancer-associated fibroblasts (CAFs)** produce excessive amounts of collagen, leading to a stiff and dense tumor stroma (desmoplasia). This dense matrix physically promotes cancer cell invasion and migration.
* **COMP (Cartilage Oligomeric Matrix Protein):** A non-collagenous protein of the ECM that helps organize other matrix components, contributing to the structural changes in the tumor.
* **ADAM12 (ADAM Metallopeptidase Domain 12):** This gene encodes a protein with both cell adhesion and protease functions. It can cleave ECM proteins, allowing cancer cells to move through the tissue more easily.
* **ITGA11 (Integrin Subunit Alpha 11):** An integrin protein that acts as a cell surface receptor. It helps fibroblasts adhere to and remodel collagen, a critical step in tumor matrix reorganization.
* **ITGBL1 (Integrin Beta Like 1):** This protein modulates cell-ECM adhesion and signaling, influencing cell migration and survival.


### Signaling and Growth Factor Regulation

These genes are involved in signaling pathways that regulate cell proliferation, differentiation, and communication within the tumor microenvironment.

* **SFRP4 (Secreted Frizzled-Related Protein 4):** A modulator of the Wnt signaling pathway, which is often dysregulated in cancer. SFRP4 can promote tumor progression and metastasis.
* **INHBA (Inhibin Beta A):** A subunit of activin A, a growth factor that can activate fibroblasts, leading to the pro-tumorigenic desmoplastic reaction.
* **CTHRC1 (Collagen Triple Helix Repeat Containing 1):** A secreted protein that promotes cell migration and ECM remodeling. It is highly expressed in CAFs and is a known contributor to cancer cell invasion and a marker for poor prognosis.

We can now visualize the spatial expression of these genes. The next section of code plots `COL1A1`'s spatial gene expression:

In [ ]:
gene_name = "COL1A1"

for i in range(len(image_elements)):
  print("Plotting: "+ image_elements[i])
  title=image_elements[i].replace("_hires_tissue_image","")
  crop0(concatenated_sdata,crs="downscale_to_hires",bbox=extents[i]).pl.render_images(image_elements[i]).pl.render_shapes(shape_elements[i],color=gene_name).pl.show(coordinate_systems="downscale_to_hires",title=title)

We observe that `COL1A1` is expressed by fibroblasts situated closer to the tumor in the `Cancer` samples. Though beyond the scope of this guide, to delve deeper into the specific biology of these fibroblasts, the next step would involve subsetting the fibroblast-containing cluster from the overall dataset. This isolated subset can then be re-clustered to further investigate differences and heterogeneity within the fibroblast populations.


# Conclusion

This guide presents a comprehensive workflow for analyzing multiple Visium HD Spatial Gene Expression datasets using Python and the Scverse ecosystem, with a focus on differential gene expression. It covers common analytical steps such as quality control, dimensionality reduction, clustering, spatial visualization, batch correction and differential gene expression.

# Appendix: Sketching workflow for multi-sample analysis

## **Sketch Downsampling**

Given the large number of bins in Visium HD data and the fact that this type of analysis involves multiple datasets, sketching or subsampling the data may be required as it significantly improves computational efficiency. This is especially true if the 8 or 16 micron binned data is used instead of the cell-segmentation based binning. In the provided code, we demonstrate how sketching can be applied to Visium HD data in Python using the cell-segmentation based binning.

We start by creating a copy of the `AnnData` object from the `SpatialData` object. A dictionary is used to store the sketch for each sample. We use `geosketch` to generate a smaller, yet representative, dataset for each sample, leveraging the PCA embedding.

While it is possible to select a constant number of bins from each sample, for our final analysis, we aim to preserve the relative proportion of each individual sample. Therefore, we define a sampling rate. This sampling rate will likely need to be empirically determined for new analyses. In this example, 25% was chosen. By maintaining a constant sampling rate, each sample contributes the same relative number of cell segmented bins to the final sketch, thus ensuring the sketched dataset's composition closely mirrors that of the full dataset.



In [ ]:
# We use the Adata object created in Section 5 below for the data sketching example.
# Run code blocks through at least Section 5 to be able to run the Sketching workflow code blocks.

# sketching
SAMPLING_RATE = 0.25

adata_for_sketch = sc.read_h5ad("preprocessed_adata.h5ad", backed="r+")

sketched_adatas = {}

for value in adata_for_sketch.obs["sample"].cat.categories:
    print(value)
    subset = adata_for_sketch.obs["sample"] == value
    subset_adata = adata_for_sketch[subset, :].to_memory()
    N = SAMPLING_RATE * subset_adata.shape[0]
    sketch_index = sketch.gs(X=subset_adata.obsm["X_pca"], N=int(N), seed=1)
    sketched_adatas[value] = subset_adata[sketch_index, :]

sketched_adata = sc.concat(sketched_adatas)

del subset, subset_adata, N, sketch_index
gc.collect()

# neighborhood and clustering resolution
RES = 0.5 # clustering resolution
NEIGHBORS = 30  # number of neighbors

MIN_DIST=0.5 #default 0.5
SPREAD=2 #default 1

sc.tl.pca(sketched_adata)

sc.pp.neighbors(sketched_adata, n_neighbors=NEIGHBORS, use_rep="X_pca",metric="correlation")
sc.tl.leiden(sketched_adata, flavor="igraph", key_added="clusters", resolution=RES,random_state=0)

sc.tl.umap(sketched_adata,min_dist=MIN_DIST, spread=SPREAD, random_state=0)

# Plot UMAP
sc.pl.umap(sketched_adata, color=["clusters"], title="Sketched UMAP by Clusters")
sc.pl.umap(sketched_adata, color=["sample"], title="Sketched UMAP by Sample")



Once the sketching process is complete, the dictionary containing the individual sketched samples is merged to form a final, combined sketch representing all samples.


##**Projection of the Sketched Results onto the Full Dataset**

Now that we have analyzed a sketch of the dataset, the clustering and embeddings (PCA and UMAP) are projected onto the full dataset. In the following code, the complete `AnnData` object is reloaded from the saved file using `Scanpy`'s `read_h5ad` function in "backed mode." This mode keeps the data on disk instead of loading it all into memory, pulling in only the necessary portions as they are needed.

To improve processing efficiency, a `chunk_size` (the number of bins to process at a time) is first defined, and the number of chunks the full dataset needs to be split into is calculated. Next, each chunk is processed using `Scanpy`'s `ingest` function, which performs several key tasks:

- It uses the PCA loadings learned from the sketched data to embed the complete dataset into the PCA space.

- It employs a winner-take-all k-nearest neighbor (kNN) classifier to assign bins in the full dataset to clusters identified in the sketched dataset.

-  It utilizes the `Transform` function from the UMAP Python package to generate the UMAP representation for the complete dataset.

Once each chunk is processed, the resulting `AnnData` objects are merged using `Scanpy`'s concat function, and the results are visualized.



In [ ]:
%%time

# Projecting the results onto the complete data set
chunk_size = 100000
num_chunks = adata_for_sketch.n_obs // chunk_size + (1 if adata_for_sketch.n_obs % chunk_size != 0 else 0)

adata_query_ingested_list = []

for i in range(num_chunks):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, adata_for_sketch.n_obs)
    adata_query_chunk = adata_for_sketch[start:end].to_memory()

    sc.tl.ingest(adata_query_chunk, sketched_adata, obs="clusters")
    adata_query_ingested_list.append(adata_query_chunk)

# Concatenate the ingested chunks
adata_query_ingested = sc.concat(adata_query_ingested_list)

adata_query_ingested.write("adata_ingested.h5ad")

del adata_query_chunk, start, end, i, num_chunks
gc.collect()

sc.pl.umap(adata_query_ingested, color=["clusters"],size=10, title="Ingested UMAP with Clusters")